###Instalation of the main package used to make the calls into the API

In [1]:
pip install -qq openai python-dotenv

###Imports needed libraries to make the code work

In [12]:
import pandas as pd
import openai
import os
from dotenv import load_dotenv
from google.colab import files
import time
import textwrap

###If you already ran the program but didn´t used all the csv rows you can load the csv to start for the first unused row.

In [24]:
output_path = "PlinianLabelCSV.csv"
df = pd.read_csv("CleanSelectedRows.csv")

###Check if the API KEY is well charged

In [15]:
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("API Key ready to use")

API Key ready to use


Methods used for labeling the texts in the csv.

In [28]:
def check_language(text, language='english'):
    prompt = (
        f"The following text may or may not be written in {language}. "
        f"Your task is to:\n"
        f"1. Detect the language.\n"
        f"2. If it's not in {language}, translate it.\n"
        f"3. Improve its clarity and correctness.\n"
        f"4. Remove any signs or messages that suggest the text was generated by an AI.\n"
        f"5. Do NOT add any comments, notes, or explanations—only return the cleaned result.\n\n"
        f"Text:\n{text}"
    )

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a language expert that improves and translates text with precision and removes AI-generated phrases."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


PlinianCoreTypes = [
    "AnnualCycles", "Audience", "Behavior", "TaxonomicDescription", "CommonNames",
    "DemographyAndThreat", "Dispersal", "Distribution", "EcologicalSignificance", "Endemic",
    "EnvironmentalEnvelope", "Feeding", "Habitats", "Interaction", "IdentificationKeys",
    "Invasiveness", "Legislation", "LifeCycle", "LifeForm",
    "ManagementAndConservation", "Migratory", "MolecularData", "PopulationBiology",
    "Reproduction", "Synonyms", "NomenclatureAndClassification", "Territory", "ThreatStatus",
    "Uses", "temporalCoverage"
]
def Plinian(text, language='english'):
    plinian_terms_list = ', '.join(PlinianCoreTypes)

    prompt = (
        f"The following text is written in {language}. Your task is to:\n"
        "1. Analyze the text and identify whether it contains information related to any of the following Plinian Core terms:\n"
        f"{plinian_terms_list}.\n\n"
        "2. For each relevant term, extract only the fragment of the text that corresponds to it.\n"
        "3. Return the results in the following format:\n\n"
        "TermName: Extracted information related to the term\n\n"
        "4. Only include terms that are actually found in the text.\n"
        "5. DO NOT add explanations, comments, or notes.\n"
        "6. Separate each term output with a blank line (double enter).\n"
        "7. DO NOT include terms that are not present in the text.\n"
        "8. DO NOT write 'Not specified', 'Not available', 'Unknown', or similar text for missing terms — JUST OMIT THEM.\n\n"
        f"Text:\n{text}"
    )

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a taxonomy assistant that extracts structured species-related information "
                    "based on Plinian Core terms."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

def label_dataframe(df, output_path):
    start_index = 0
    if 'Cleaned_Text' not in df.columns:
        df['Cleaned_Text'] = pd.NA
    if 'PlinianCore_Labels' not in df.columns:
        df['PlinianCore_Labels'] = pd.NA

    if os.path.exists(output_path):
        try:
            existing = pd.read_csv(output_path)
            start_index = len(existing[existing['Cleaned_Text'].notna()])
            df['Cleaned_Text'] = existing.get('Cleaned_Text', pd.NA)
            df['PlinianCore_Labels'] = existing.get('PlinianCore_Labels', pd.NA)
        except Exception as e:
            print(f"Error loading existing file: {e}")
            pass

    total_rows = len(df) - start_index
    print(f"\nThere are {total_rows} rows left to process starting from index {start_index}.")
    while True:
        try:
            max_rows = int(input("How many rows would you like to process? (Enter a positive integer): "))
            if max_rows <= 0:
                print("Please enter a number greater than 0.")
            else:
                break
        except ValueError:
            print("Invalid input. Please enter an integer.")

    end_index = min(start_index + max_rows, len(df))

    for i in range(start_index, end_index):
        text = str(df.at[i, 'Text'])
        language = str(df.at[i, 'Language']).lower()

        if pd.isna(text) or text.strip() == "":
            print(f"Skipping empty row {i}")
            continue

        print(f"\nProcessing row {i} ({language})...")

        try:
            cleaned_text = check_language(text, language)
            labels = Plinian(cleaned_text, 'english')
            df.at[i, 'Cleaned_Text'] = cleaned_text
            df.at[i, 'PlinianCore_Labels'] = labels
        except Exception as e:
            print(f"Error in row {i}: {e}")
            df.at[i, 'Cleaned_Text'] = None
            df.at[i, 'PlinianCore_Labels'] = None

        time.sleep(1.2)

    df.to_csv(output_path, index=False)
    print("\nTagging and cleaning completed. File saved.")

###Run this cell to label the csv you added.

In [29]:
label_dataframe(df, output_path)
files.download("PlinianLabelCSV.csv")


There are 302 rows left to process starting from index 0.
How many rows would you like to process? (Enter a positive integer): 302

Processing row 0 (english)...

Processing row 1 (english)...


KeyboardInterrupt: 

Function for reviewing the labels and the text in a cleaner way

In [20]:
def review_labeled_rows(df):
    for i in range(len(df)):
        original = str(df['Text'].iloc[i])
        language = str(df['Language'].iloc[i])
        cleaned = str(df['Cleaned_Text'].iloc[i])
        PlinianCore_Labels = str(df['PlinianCore_Labels'].iloc[i])

        if pd.isna(cleaned) or original.strip() == "":
            continue  # Skip if text is empty or not yet cleaned

        print(f"\n--- Row {i} ---")

        print("\nOriginal Text:\n")
        print(textwrap.fill(original, width=100))

        print("\nLanguage:\n")
        print(language)

        print("\nCleaned Text:\n")
        print(textwrap.fill(cleaned, width=100))

        print("\nNote:")
        print("Maybe the original text wasn’t in the language specified; it was cleaned and translated to English by the API.")

        print("\nPlinian Core Labels:\n")
        print(textwrap.fill(PlinianCore_Labels, width=100))

        while True:
            cont = input("\nDo you want to continue to the next row? (y/n): ").strip().lower()
            if cont in ['y', 'n']:
                break
            print("Please enter 'y' or 'n'.")

        if cont == 'n':
            print("Stopping review.")
            break

###Run this cell if you want to check how to label went

In [26]:
df = pd.read_csv("PlinianLabelCSV.csv")
review_labeled_rows(df)


--- Row 0 ---

Original Text:

Radulae of many species have been illustrated (O Donoghue, 1924; Hand Steinberg, 1955; Gonor, 1961;
Lance, 1962; Marcus Marcus, 1970b; Ferreira Bertsch, 1975; Gascoigne, 1975; Farmer, 1980; Bleakney,
1989, 1990; Behrens, 1991b; Valdés Camacho-Garcia, 2000). Jensen (1980, 1993, 1996, 1997)
hypothesized that tooth shape is directly related to food type. Jensen s results may assist in
determining the diets of the poorly studied species; such extrapolation would be a useful tool for
future study of uncommon sacoglossans. Furthermore, Bleakney (1989, 1990) and Jensen (1996, 1997)
have reported intraspecific variation in radular tooth morphology in two species on different diets;
it would be intriguing to know whether this phenomenon also occurs in Aplysiopsis enteromorphae,
Elysia hedgpethi, and other northeastern Pacific species that feed on two or more genera of algae.

Language:

English

Cleaned Text:

The radulae of many species have been illustrated (O’

KeyboardInterrupt: Interrupted by user